# 注意力算子优化

第 4 章用 RMSNorm 跑通了融合算子的完整流程——一行 converter 替换，profiler 验证，wall-time 对比。本章把同样的分析方法用到 attention 上。Attention 的优化空间比 RMSNorm 更大：序列长度翻倍，score 矩阵的 $O(S^2)$ 逻辑规模翻四倍。FlashAttention、GQA、稀疏注意力、VarLen 从不同维度切入，各自的收益边界需要实验来定。

---

## 教程进度回顾

| 章节 | 内容 | 状态 |
|------|------|------|
| 第 1 章 | SFT 概念 + Wordle 任务 | ✅ 已完成 |
| 第 2 章 | TorchTitan 框架 + 环境配置 | ✅ 已完成 |
| 第 3 章 | 数据准备 + 基线训练 + 推理评测 | ✅ 已完成 |
| 第 4 章 | 融合算子 + Profiling 优化 | ✅ 已完成 |
| **第 5 章** | **Attention 算子：从概念到 kernel** | ← 当前 |

---

## 本章目标

完成本章后，你将能够：

- 区分 SDPA、FlashAttention、GQA、稀疏注意力、document mask 与 VarLen Attention 分别改变了哪一层；
- 用 correctness、wall-time 和 profiler trace 为 attention kernel 设计受控对照；
- 追踪 document boundary metadata 从 DataLoader 到 NPU attention backend 的传递路径；
- 为第 6 章的 SFT 变长注意力优化建立概念和实验基线。

---

## 问题：Attention 优化不是单一答案

第 3 章基线训练用的是 PyTorch SDPA——一行 API 调用。但这行 API 后面发生了什么、当前软硬件下实际跑的是哪个 kernel、GQA 和 document mask 各自省了什么、VarLen 跳过跨文档计算能快多少——这些问题不拆开验证，就没法判断下一步该优化什么。

本章不引入新的优化代码。它先把概念基线立起来，再逐条用 wall-time 和 trace 对照，最后把数据路径（document boundary metadata 从哪来）和 backend 路径（怎么交给 NPU kernel）接在一起。这样第 6 章做 SFT 变长注意力优化时，每一个设计决策都有实验依据。

### 分析路线

- **概念层**（05.02）：SDPA dispatch → FlashAttention → GQA → 稀疏注意力 → SFT document mask → VarLen Attention。明确每种技术改的是哪一层、不改什么。
- **实验层**（05.03）：为每条理论设计对照实验——eager vs SDPA vs 直接 V3、`Nkv=16` vs `Nkv=8`、causal vs document mask、dense vs VarLen sweep。wall-time 回答“快不快”，profiler trace 回答“跑了什么”。
- **工程层**（05.04 + 05.05）：追踪 dataloader 如何让每条样本的位置编号重新从 0 开始，以及 trainer 如何据此生成 `NPUVarlenAttention` 使用的 `cu_seq`。

---

## 前置条件

- 第 2 章环境已配置，`torch_npu` 可正常 import，Ascend NPU 可用。
- 第 3 章基线训练已完成（理解 SDPA 在训练中的角色）。
- 第 4 章融合算子方法论已了解（本章沿用同样的 wall-time + trace 验证框架）。

---

## 本章结构

| Notebook | 内容 |
|---|---|
| [05.01](05.01_chapter_intro.ipynb) | 章节介绍（本节） |
| [05.02](05.02_attention_kernels.ipynb) | 概念基线：SDPA → FlashAttention → GQA → 稀疏注意力 → document mask → VarLen |
| [05.03](05.03_attention_kernel_benchmarking.ipynb) | 算子验证：五组对照实验，wall-time 和 trace 逐项检验 |
| [05.04](05.04_torchtitan_dataloader.ipynb) | 数据路径：从样本位置重置到 `cu_seq` 的完整链路 |
| [05.05](05.05_torchtitan_attention_ops.ipynb) | Backend 路径：attention config → dispatch → NPU VarLen |
| [05.06](05.06_chapter_practice.ipynb) | 章节练习 |

下一节先建立概念基线——不是记算子名，是分清每种技术改变了哪一层。


In [ ]:
!cat ./answer/05.01_answer.txt
